# Lv3 실전 예상문제 1회 — 상품 데이터·모델·LLM

**창작 연습문제입니다. 실제 Level 3 문제·배점·모델·데이터·제출 방식·성능 기준은 미공개입니다.** 연습 조건은 오프라인 240분, 데이터 40점·모델 35점·LLM 활용 25점입니다. 지문을 읽고 작성 셀을 완성하세요. 각 문항은 독립적입니다. 폴더의 정답_참고는 풀이 전에 열지 마세요. 권장: 70분 / 85분 / 65분 / 검토 20분.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / 'practice_support.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('모의문제 폴더 또는 정답_참고 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(ROOT.resolve()))
import copy, random, numpy as np, pandas as pd, torch
from torch import nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from practice_support import load_q1,load_q2,load_q3,rmse,MockTokenizer,MockCausalLM
torch.set_num_threads(2); random.seed(42); np.random.seed(42); torch.manual_seed(42)

## 1. 데이터 결합과 회귀 기준 모델 (40점)
`products`는 train/valid/test 상품, `events`는 상품별 과거 views/clicks, `labels`는 학습·검증 상품의 purchases/impressions입니다. test 정답은 제공하지 않습니다.

(a) [8] 이벤트를 상품별 합산해 왼쪽 상품 순서대로 1:1 병합하고 ctr=clicks/views를 생성하세요. 모든 views 합은 양수입니다.
(b) [6] labels를 병합해 target=purchases/impressions. test target은 결측입니다. 정답에 사용된 두 열을 입력에 넣지 않습니다.
(c) [10] `[price,ram,storage,views,clicks,ctr]` 순서로 입력을 선택합니다. 결측은 train 중앙값, 표준화는 train 기준으로 fit하고 valid/test는 transform만 적용하세요.
(d) [8] LinearRegression을 train으로 학습, valid/test 예측을 0~1로 제한합니다.
(e) [8] valid RMSE와 test의 product_id/prediction DataFrame을 원래 test 순서로 반환합니다. 참고 목표(창작) valid RMSE ≤ 0.012.


In [ ]:
def solve_data(products, events, labels):
 # 반환: model, imputer, scaler, valid_rmse, test_predictions를 담은 dict
 raise NotImplementedError('1번 작성')

In [ ]:
products,events,labels=load_q1(); q1_result=solve_data(products,events,labels)
p=q1_result['test_predictions']; assert p.product_id.tolist()==products.loc[products['split'].eq('test'),'product_id'].tolist()
assert p.prediction.between(0,1).all(); print('검증 RMSE',q1_result['valid_rmse'])

## 2. 신경망 회귀 및 조기 종료 (35점)
1번과 독립적으로 전처리된 `X_train`, `X_valid`, `y_train`, `y_valid`를 제공합니다.

(a) [7] `Linear(입력차원,32) → ReLU → Linear(32,1) → Sigmoid`.
(b) [10] DataLoader 배치32, Adam lr=0.01, MSE. `zero_grad`, 순전파, 손실, `backward`, `step` 순으로 수행하고 예측과 정답을 `(B,)`로 맞추세요.
(c) [10] 최대 100 epoch. 각 epoch마다 eval/no_grad 검증 RMSE가 낮아지면 가중치를 복사하고 12회 연속 개선되지 않으면 중단합니다. 최선 가중치를 복원합니다.
(d) [4] 최선 모델·검증 예측·RMSE·실행 epoch를 반환하고 seed=42를 설정하세요. 참고 목표(창작): RMSE ≤ 0.020.
(e) [4] 검증 오차를 사용하는 이유를 1~2문장으로 `answer_q2`에 작성하세요.


In [ ]:
def train_model(data):
 # 반환: model, pred_valid, valid_rmse, epochs를 담은 dict
 raise NotImplementedError('2번 작성')
answer_q2='' # (e)

In [ ]:
ready=load_q2(); q2_result=train_model(ready)
assert q2_result['pred_valid'].shape==ready['y_valid'].shape
print('검증 RMSE',q2_result['valid_rmse'],'epochs',q2_result['epochs'])

## 3. 근거 검색과 오프라인 LLM 호출 (25점)
`doc_vectors`는 `(문서 수,D)`, `query_vector`는 `(D,)`인 정규화되지 않은 벡터입니다. 행 순서는 documents와 같으며 노름은 0이 아닙니다.

(a) [8] NumPy 코사인 유사도 전체와 내림차순 상위 `min(k,문서 수)` 위치를 구하세요. 문서 수를 고정하지 마세요.
(b) [6] 상위 문서 id·텍스트와 질문을 포함하고, 근거가 없으면 모른다고 답하라는 프롬프트를 작성하세요.
(c) [7] tokenizer/model을 받아 토큰화 후 최대 새 토큰32개를 `do_sample=False`로 생성하고 입력 이후 새 토큰만 decode하세요. attention_mask, model.device, eval/no_grad, pad_token_id 없으면 eos_token_id를 사용합니다. 토큰화는 truncation=True, max_length=128.
(d) [4] 각 질문의 상위 2개 id와 hit@2를 계산하세요. 정답 topic이 검색 문서 중 하나와 같으면 성공입니다. topic은 평가에만 사용하고 검색 입력에 쓰지 않습니다.

확인 셀의 Mock은 인터페이스 검사용 가짜 모델이며 LLM 답변 품질을 평가하지 않습니다. 실제 로컬 모델 선택 실행법은 Reference를 참고하세요.


In [ ]:
def retrieve(query_vector,doc_vectors,k):
 raise NotImplementedError('3(a)')
def make_prompt(question,documents,indices):
 raise NotImplementedError('3(b)')
def generate_answer(prompt,tokenizer,model):
 raise NotImplementedError('3(c)')
def evaluate_retrieval(documents,queries,doc_vectors,query_vectors):
 raise NotImplementedError('3(d)')

In [ ]:
documents,queries,doc_vectors,query_vectors=load_q3()
ids,hit=evaluate_retrieval(documents,queries,doc_vectors,query_vectors); print('검색 결과:',ids,'hit@2:',hit)
_,indices=retrieve(query_vectors[0],doc_vectors,2); prompt=make_prompt(queries[0]['text'],documents,indices)
print('가짜 모델 인터페이스 확인:',generate_answer(prompt,MockTokenizer(),MockCausalLM()))

## 제출 전 확인
답안과 실행 결과를 저장하세요. Q1 test 예측은 선택적으로 CSV로 저장할 수 있습니다. 실제 Level 3 제출 형식은 미공개입니다.